# Step 5: reliability metrics (RQ1)

Run this **after Step 4 has finished** (all three products downloaded and the inventory done). It reads the L3 files from your Drive, matches them with the drifters from Step 2, and computes Accuracy, Availability and Error Consistency, then the product scores. It does not touch forecast errors.

You need: the newest `nbob_sst_study` zip and your Copernicus Marine login. Usually 15–40 minutes in total. Run the cells from top to bottom (Shift+Enter).

## 1. Clean start and install

In [ ]:
%cd /content
!rm -rf /content/nbob_sst_study /content/*.zip
!pip -q install xarray netCDF4 h5netcdf siphon copernicusmarine pandas requests

## 2. Upload the newest `nbob_sst_study` zip

In [ ]:
from google.colab import files
uploaded = files.upload()
zipname = list(uploaded.keys())[0]
!unzip -o -q "$zipname" -d /content
%cd /content/nbob_sst_study
!cat VERSION

**Check:** the cell above must print `2026-10-11d` (or later).

## 3. Connect Drive and check the inputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
L3 = '/content/drive/MyDrive/nbob_step4_l3'
IQ = '/content/drive/MyDrive/nbob_step2_iquam/box_monthly'
OUT = '/content/drive/MyDrive/nbob_step5'
os.makedirs(OUT, exist_ok=True)
for p in ['metopb', 'aqua', 'terra']:
    print(p, len(glob.glob(f'{L3}/{p}/*/*_nbob.nc')), 'L3 files')
print(len(glob.glob(f'{IQ}/box_*.csv')), 'iQuam month files (expected 36)')

**Check:** each product should show roughly 2,000 or more files, and iQuam 36 files. If a product shows 0, finish Step 4 first.

## 4. Copernicus Marine login (for the daily L4 SST)

In [ ]:
import copernicusmarine, getpass
user = input('Copernicus Marine username: ')
pw = getpass.getpass('Password: ')
copernicusmarine.login(username=user, password=pw)

## 5. Main run
Downloads the daily L4 for the station area and the GEBCO_2023 bathymetry once, then processes all L3 files.

In [ ]:
!python step5_reliability.py --l3 "$L3" --iquam "$IQ" --out "$OUT" 2>&1 | grep -v Warning | tee "$OUT/log_main.txt"

## 6. Two checks from the analysis plan
Same run with quality level 5, and with a 10 km station radius. They reuse the L4 and bathymetry from cell 5.

In [ ]:
!python step5_reliability.py --l3 "$L3" --iquam "$IQ" --l4 "$OUT/l4_daily_stationarea.nc" --bathy "$OUT/bathymetry_nbob.nc" --ql-min 5 --out "$OUT/check_ql5" 2>&1 | grep -v Warning | tail -30
!python step5_reliability.py --l3 "$L3" --iquam "$IQ" --l4 "$OUT/l4_daily_stationarea.nc" --bathy "$OUT/bathymetry_nbob.nc" --radius-km 10 --out "$OUT/check_r10" 2>&1 | grep -v Warning | tail -30

## 7. Download the results
Send `step5_results.zip` back in the chat.

In [ ]:
!cd "$OUT" && zip -q -r /content/step5_results.zip *.csv *.txt check_ql5/*.csv check_ql5/*.txt check_r10/*.csv check_r10/*.txt
files.download('/content/step5_results.zip')

## If something fails
Copy the last 20 lines of the red error and send them in the chat.